# Reverse-engineering the scores

Flip the model: predict `EXT_SOURCE` from the internal features. High reconstruction R² means the purchased score is largely rebuildable from data the lender already owns; low R² means it carries information from outside the lender's walls.

In [ ]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score
from src.data import load_selected, RAW

X, y = load_selected()
ext = [c for c in X.columns if c.lower().startswith("ext_") or "ext_source" in c.lower()]
internal = [c for c in X.columns if c not in ext]
raw_ext = pd.read_csv(RAW / "application_train.csv",
                      usecols=["SK_ID_CURR", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]).set_index("SK_ID_CURR").reindex(X.index)

def reg():
    return lgb.LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=31, subsample=0.8,
                             colsample_bytree=0.7, subsample_freq=1, n_jobs=-1, verbose=-1)
raw_ext.notna().mean().round(3)

## How reconstructable is each score?

In [ ]:
kf = KFold(5, shuffle=True, random_state=42)
for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]:
    m = raw_ext[c].notna().to_numpy()
    pred = cross_val_predict(reg(), X.loc[m, internal], raw_ext[c][m], cv=kf)
    print(f"{c}:  R2 {r2_score(raw_ext[c][m], pred):+.3f}   (n={int(m.sum())}, coverage {m.mean():.2f})")

## What is the score made of?

In [ ]:
import shap
c = "EXT_SOURCE_2"
m = raw_ext[c].notna()
model = reg().fit(X.loc[m, internal], raw_ext[c][m])
samp = X.loc[m, internal].sample(min(5000, int(m.sum())), random_state=0)
sv = model.booster_.predict(samp, pred_contrib=True)[:, :-1]
pd.Series(np.abs(sv).mean(0), index=internal).sort_values(ascending=False).head(15)